# 05 - Missing Value Analysis & Imputation

## Objective

Learn how to identify, understand and treat missing values in Marketing Mix Modeling datasets.

> In production, blindly filling missing values can introduce bias into the model. The choice of imputation depends on why the data is missing.


In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.impute import SimpleImputer, KNNImputer

ROOT = Path.cwd()
DATA = ROOT / "data" / "processed" / "marketing_mix_cleaned.csv"

df = pd.read_csv(DATA, parse_dates=["Week"])
df.head()


## 1. Missing Data Mechanisms

- **MCAR** (Missing Completely At Random)
- **MAR** (Missing At Random)
- **MNAR** (Missing Not At Random)

Understanding the mechanism helps determine the appropriate treatment.


## 2. Missing Value Summary

In [ ]:

missing = (
    df.isna()
      .sum()
      .sort_values(ascending=False)
      .rename("Missing_Count")
      .to_frame()
)

missing["Missing_%"] = (missing["Missing_Count"]/len(df)*100).round(2)
display(missing)


## 3. Missing Value Visualization

In [ ]:

plt.figure(figsize=(12,4))
plt.bar(missing.index.astype(str), missing["Missing_Count"])
plt.xticks(rotation=90)
plt.title("Missing Values by Feature")
plt.tight_layout()
plt.show()


## 4. Median Imputation (Numeric Features)

In [ ]:

numeric_cols = df.select_dtypes(include=np.number).columns

median_imputer = SimpleImputer(strategy="median")
median_df = df.copy()
median_df[numeric_cols] = median_imputer.fit_transform(median_df[numeric_cols])

print("Remaining missing values:", median_df.isna().sum().sum())


## 5. KNN Imputation (Demonstration)

In [ ]:

knn_df = df.copy()

knn = KNNImputer(n_neighbors=5)
knn_df[numeric_cols] = knn.fit_transform(knn_df[numeric_cols])

print("Remaining missing values:", knn_df.isna().sum().sum())


## 6. Compare Imputation Methods

In [ ]:

comparison = pd.DataFrame({
    "Original": df[numeric_cols].isna().sum(),
    "Median": median_df[numeric_cols].isna().sum(),
    "KNN": knn_df[numeric_cols].isna().sum()
})

display(comparison.head(15))


## 7. Business Considerations

In [ ]:

recommendations = pd.DataFrame({
    "Scenario":[
        "Media Spend Missing",
        "Sales Missing",
        "Weather Missing",
        "Holiday Flag Missing",
        "Price Missing"
    ],
    "Preferred Action":[
        "Investigate source before imputing",
        "Do not impute blindly; verify source",
        "Use nearest weather station or interpolation",
        "Reconstruct from calendar",
        "Median or business rule"
    ]
})

display(recommendations)


## 8. Save Imputed Dataset

In [ ]:

OUTPUT = ROOT / "data" / "processed" / "marketing_mix_imputed.csv"
median_df.to_csv(OUTPUT, index=False)
print("Saved:", OUTPUT)


# Key Takeaways

- Always understand *why* data is missing before choosing an imputation method.
- Median imputation is robust for skewed numeric features.
- KNN imputation can preserve local structure but is more computationally expensive.
- Critical business fields (e.g., sales or media spend) should often be investigated rather than automatically filled.
